# Café Data Analysis — Part 3
In Part 3 we focus on one targeted question using only `cafe_clean_lines.csv` from Part 1: **Do Red Bull promotions actually increase daily Red Bull revenue after accounting for how busy the café is (weekends) and the January 2011 price increase, and what kind of customer order is most likely to buy a deal?**

## Table of Contents
- [Part 3 - Red Bull Promotions, Traffic & Price Controls, and Deal-Buying Orders](#part-3)
  - [Step 1 - Setup & Analytical Questions](#step-1)
    - [Step 1.1 - Questions and hypotheses](#step-1-1)
    - [Step 1.2 - Libraries and cleaned line data](#step-1-2)
  - [Step 2 - Red Bull Revenue After Controlling for Weekends and the January 2011 Price Increase](#step-2)
    - [Step 2.1 - Building the daily Red Bull and café traffic dataset from lines](#step-2-1)
    - [Step 2.2 - Promo vs. non-promo revenue across weekend traffic and January 2011 pricing](#step-2-2)
    - [Step 2.3 - Regression check on Red Bull promotion lift](#step-2-3)
  - [Step 3 - Which Customer Orders Are Most Likely to Buy a Deal?](#step-3)
    - [Step 3.1 - Aggregating bill-level orders from cafe_clean_lines.csv](#step-3-1)
    - [Step 3.2 - Deal attachment by order composition, basket size, daypart, and order value](#step-3-2)
    - [Step 3.3 - Modeling deal likelihood across customer orders](#step-3-3)
  - [Step 4 - Key Takeaways & Limitations](#step-4)
    - [Step 4.1 - Summary of findings](#step-4-1)
    - [Step 4.2 - Data limitations](#step-4-2)

<a id="part-3"></a>
## Part 3 - Red Bull Promotions, Traffic & Price Controls, and Deal-Buying Orders

<a id="step-1"></a>
### Step 1 - Setup & Analytical Questions

<a id="step-1-1"></a>
#### Step 1.1 - Questions and hypotheses
In **Part 1**, we saw that Red Bull items account for the largest share of promotional volume (`RED BULL 2+1` through early January 2011, shifting to `2`, `3`, and `5 RED BULL` multi-packs in February–March 2011), while café traffic peaks on **weekends** and prices rose in **January 2011** (base `RED BULL ENERGY DRINK` moving from `125` to `150` on the new menu from 7 January 2011).

Part 3 investigates two connected questions using `cafe_clean_lines.csv`:
1. **Do Red Bull promotions actually increase daily Red Bull revenue after accounting for how busy the café is (weekends) and the January 2011 price increase?**
2. **What kind of customer order is most likely to buy a deal?**

<a id="step-1-2"></a>
#### Step 1.2 - Libraries and cleaned line data
We load only `cafe_clean_lines.csv` produced at the end of Part 1 (Step 7.4) and derive daily and order-level tables directly from those item lines.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

lines = pd.read_csv("cafe_clean_lines.csv")
lines["Date"] = pd.to_datetime(lines["Date"])
lines["Business day"] = pd.to_datetime(lines["Business day"])

print(f"Lines loaded: {lines.shape[0]:,} rows x {lines.shape[1]} columns")

<a id="step-2"></a>
### Step 2 - Red Bull Revenue After Controlling for Weekends and the January 2011 Price Increase

<a id="step-2-1"></a>
#### Step 2.1 - Building the daily Red Bull and café traffic dataset from lines
From `cafe_clean_lines.csv`, we group `Base product == "RED BULL ENERGY DRINK"` lines by `Business day` to get daily Red Bull revenue and units, and group all lines by `Business day` to count unique bills (`day_orders`) and total daily revenue (`day_revenue`), along with `Weekend` and `post_jan1` (`Business day >= 2011-01-01`).

In [ ]:
rb_lines = lines[lines["Base product"] == "RED BULL ENERGY DRINK"].copy()
rb_lines["is_rb_promo"] = rb_lines["Promo"] != "No promo"

day_traffic = (
    lines.groupby("Business day")
    .agg(
        Weekend=("Weekend", "first"),
        Weekday=("Weekday", "first"),
        Menu=("Menu", "first"),
        Special_day=("Special day", "first"),
        day_orders=("Bill Number", "nunique"),
        day_revenue=("Total", "sum"),
    )
    .reset_index()
)

rb_daily = (
    rb_lines.groupby("Business day")
    .agg(
        rb_revenue=("Total", "sum"),
        rb_units=("Quantity", "sum"),
        rb_promo_revenue=("Total", lambda s: s[rb_lines.loc[s.index, "is_rb_promo"]].sum()),
        rb_promo_units=("Quantity", lambda s: s[rb_lines.loc[s.index, "is_rb_promo"]].sum()),
        rb_promo_lines=("Quantity", lambda s: int(rb_lines.loc[s.index, "is_rb_promo"].sum())),
    )
    .reset_index()
)

rb_panel = (
    day_traffic.merge(rb_daily, on="Business day", how="left")
    .fillna({
        "rb_revenue": 0.0,
        "rb_units": 0.0,
        "rb_promo_revenue": 0.0,
        "rb_promo_units": 0.0,
        "rb_promo_lines": 0,
    })
    .sort_values("Business day")
    .reset_index(drop=True)
)

rb_panel["has_rb_promo"] = (rb_panel["rb_promo_lines"] > 0).astype(int)
rb_panel["weekend_int"] = rb_panel["Weekend"].astype(int)
rb_panel["post_jan1"] = (rb_panel["Business day"] >= pd.Timestamp("2011-01-01")).astype(int)
rb_panel["non_special"] = (rb_panel["Special_day"] == "No").astype(int)

rb_panel[["Business day", "Weekday", "Weekend", "day_orders", "has_rb_promo", "rb_units", "rb_revenue", "post_jan1"]].head(10)

<a id="step-2-2"></a>
#### Step 2.2 - Promo vs. non-promo revenue across weekend traffic and January 2011 pricing
We compare **Red Bull promo days vs. non-promo days** inside each **January 2011 period × Weekend/Weekday** bucket to see whether active promotions lift daily Red Bull revenue after accounting for weekend café traffic and post-January 2011 prices.

In [ ]:
cell_summary = (
    rb_panel.groupby(["post_jan1", "weekend_int", "has_rb_promo"])
    .agg(
        days=("Business day", "count"),
        mean_rb_rev=("rb_revenue", "mean"),
        median_rb_rev=("rb_revenue", "median"),
        mean_rb_units=("rb_units", "mean"),
        mean_day_orders=("day_orders", "mean"),
    )
    .round(2)
)
print(cell_summary.to_string())

rb_panel["regime_label"] = (
    np.where(rb_panel["post_jan1"] == 1, "Post-Jan 2011", "Pre-Jan 2011")
    + " | "
    + np.where(rb_panel["weekend_int"] == 1, "Weekend", "Weekday")
)
regime_order = [
    "Pre-Jan 2011 | Weekday",
    "Pre-Jan 2011 | Weekend",
    "Post-Jan 2011 | Weekday",
    "Post-Jan 2011 | Weekend",
]

fig, ax = plt.subplots(figsize=(10.5, 5.2))
sns.barplot(
    data=rb_panel,
    x="regime_label",
    y="rb_revenue",
    hue="has_rb_promo",
    order=regime_order,
    errorbar="ci",
    palette=["#8c96a5", "#1f77b4"],
    ax=ax,
)
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ["No Red Bull Promo", "Red Bull Promo Active"], title="Promotion Status")
ax.set_title("Mean Daily Red Bull Revenue: Promo vs. No Promo Across Traffic & Jan 2011 Pricing")
ax.set_xlabel("Price Period & Day Type")
ax.set_ylabel("Daily Red Bull Revenue (95% CI)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

<a id="step-2-3"></a>
#### Step 2.3 - Regression check on Red Bull promotion lift
We fit OLS models estimating daily Red Bull revenue (`rb_revenue`) on `has_rb_promo` while controlling for `weekend_int`, `day_orders`, `post_jan1`, and `non_special`, including interaction terms to test whether promo lift differs on weekends or after January 1, 2011.

In [ ]:
m_add = smf.ols(
    "rb_revenue ~ has_rb_promo + weekend_int + post_jan1 + day_orders + non_special",
    data=rb_panel,
).fit()
print("=== Additive Model ===")
print(m_add.summary().tables[1])
print(f"R2: {m_add.rsquared:.4f} | Adj R2: {m_add.rsquared_adj:.4f}\n")

m_int = smf.ols(
    "rb_revenue ~ has_rb_promo * weekend_int + has_rb_promo * post_jan1 + day_orders + non_special",
    data=rb_panel,
).fit()
print("=== Interaction Model (Promo x Weekend, Promo x Post-Jan 2011) ===")
print(m_int.summary().tables[1])
print(f"R2: {m_int.rsquared:.4f} | Adj R2: {m_int.rsquared_adj:.4f}")

<a id="step-3"></a>
### Step 3 - Which Customer Orders Are Most Likely to Buy a Deal?

<a id="step-3-1"></a>
#### Step 3.1 - Aggregating bill-level orders from cafe_clean_lines.csv
Because we are using only `cafe_clean_lines.csv`, we roll up item rows by `Bill Number` to inspect order composition, order size, order value, and deal attachment (`has_any_promo` and `has_rb_promo`).

In [ ]:
lines_work = lines.copy()
lines_work["is_promo"] = lines_work["Promo"] != "No promo"
lines_work["is_rb_promo"] = (lines_work["Base product"] == "RED BULL ENERGY DRINK") & lines_work["is_promo"]

orders_from_lines = (
    lines_work.groupby("Bill Number")
    .agg(
        Business_day=("Business day", "first"),
        Weekday=("Weekday", "first"),
        Weekend=("Weekend", "first"),
        Daypart=("Daypart", "first"),
        Hour=("Hour", "first"),
        Special_day=("Special day", "first"),
        Menu=("Menu", "first"),
        Bill_series=("Bill series", "first"),
        Products=("Item Desc", "nunique"),
        Units=("Quantity", "sum"),
        Order_value=("Total", "sum"),
        Has_FOOD=("Category", lambda s: (s == "FOOD").any()),
        Has_BEVERAGE=("Category", lambda s: (s == "BEVERAGE").any()),
        Has_TOBACCO=("Category", lambda s: (s == "TOBACCO").any()),
        Has_LIQUOR=("Category", lambda s: (s == "LIQUOR").any()),
        Has_WINES=("Category", lambda s: (s == "WINES").any()),
        Has_promo=("is_promo", "any"),
        Has_rb_promo=("is_rb_promo", "any"),
    )
    .reset_index()
)

print(f"Orders aggregated from lines: {len(orders_from_lines):,}")

<a id="step-3-2"></a>
#### Step 3.2 - Deal attachment by order composition, basket size, daypart, and order value
We compare deal rates (`Has_promo` and `Has_rb_promo`) across **category basket mix**, **order item count**, **daypart**, and **order value quartiles**.

In [ ]:
ord_df = orders_from_lines.copy()
ord_df["items_bin"] = pd.cut(
    ord_df["Products"],
    bins=[0, 1, 2, 3, 5, 100],
    labels=["1 item", "2 items", "3 items", "4-5 items", "6+ items"],
)
ord_df["value_quartile"] = pd.qcut(
    ord_df["Order_value"], q=4, labels=["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"]
)
ord_df["basket_mix"] = np.where(
    ord_df["Has_FOOD"] & ord_df["Has_BEVERAGE"] & ord_df["Has_TOBACCO"], "Food + Drink + Hookah",
    np.where(
        ord_df["Has_FOOD"] & ord_df["Has_BEVERAGE"], "Food + Drink",
        np.where(
            ord_df["Has_BEVERAGE"] & ord_df["Has_TOBACCO"], "Drink + Hookah",
            np.where(
                ord_df["Has_FOOD"] & ord_df["Has_TOBACCO"], "Food + Hookah",
                np.where(ord_df["Products"] == 1, "Single-item order", "Other multi-item"),
            ),
        ),
    ),
)

mix_summary = (
    ord_df.groupby("basket_mix", observed=False)
    .agg(
        orders=("Bill Number", "count"),
        any_deal_pct=("Has_promo", lambda s: s.mean() * 100),
        rb_deal_pct=("Has_rb_promo", lambda s: s.mean() * 100),
        avg_order_value=("Order_value", "mean"),
    )
    .sort_values("any_deal_pct", ascending=True)
)
print(mix_summary.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))

axes[0].barh(mix_summary.index, mix_summary["any_deal_pct"], color="#2b5c8f")
for idx, (_, row) in enumerate(mix_summary.iterrows()):
    axes[0].text(
        row["any_deal_pct"] + 0.25,
        idx,
        f"{row['any_deal_pct']:.1f}% (n={int(row['orders']):,})",
        va="center",
        fontsize=8.5,
    )
axes[0].set_xlim(0, mix_summary["any_deal_pct"].max() * 1.28)
axes[0].set_xlabel("Share of Orders with a Deal (%)")
axes[0].set_title("Deal Rate by Customer Order Composition")

items_summary = ord_df.groupby("items_bin", observed=False)["Has_promo"].mean() * 100
axes[1].plot(items_summary.index.astype(str), items_summary.values, marker="o", linewidth=2, color="#d95f02")
for x_i, val in enumerate(items_summary.values):
    axes[1].text(x_i, val + 0.25, f"{val:.1f}%", ha="center", fontsize=8.5)
axes[1].set_ylim(0, items_summary.max() * 1.25)
axes[1].set_xlabel("Distinct Products in Order")
axes[1].set_ylabel("Share of Orders with a Deal (%)")
axes[1].set_title("Deal Likelihood Across Order Sizes")

plt.tight_layout()
plt.show()

<a id="step-3-3"></a>
#### Step 3.3 - Modeling deal likelihood across customer orders
We fit a logistic regression model across all orders (`Has_promo`) controlling for `Products`, `Order_value`, `weekend_int`, `post_jan1`, and category flags (`Has_FOOD`, `Has_BEVERAGE`, `Has_TOBACCO`, `Has_LIQUOR`, `Has_WINES`).

In [ ]:
logit_df = ord_df.assign(
    has_promo_int=ord_df["Has_promo"].astype(int),
    weekend_int=ord_df["Weekend"].astype(int),
    post_jan1=(ord_df["Business_day"] >= pd.Timestamp("2011-01-01")).astype(int),
    has_food_int=ord_df["Has_FOOD"].astype(int),
    has_bev_int=ord_df["Has_BEVERAGE"].astype(int),
    has_tob_int=ord_df["Has_TOBACCO"].astype(int),
    has_liq_int=ord_df["Has_LIQUOR"].astype(int),
    has_wine_int=ord_df["Has_WINES"].astype(int),
)

logit_model = smf.logit(
    "has_promo_int ~ Products + Order_value + weekend_int + post_jan1 + has_food_int + has_bev_int + has_tob_int + has_liq_int + has_wine_int",
    data=logit_df,
).fit(disp=False)

or_table = pd.DataFrame({
    "coef": logit_model.params,
    "odds_ratio": np.exp(logit_model.params),
    "p_value": logit_model.pvalues,
}).round(4)
print(or_table.to_string())

<a id="step-4"></a>
### Step 4 - Key Takeaways & Limitations

<a id="step-4-1"></a>
#### Step 4.1 - Summary of findings
- **Red Bull promotions add positive daily Red Bull revenue even after controls**: controlling for weekend traffic, daily café order count, special days, and the January 2011 price increase, active Red Bull promotion days remain associated with higher daily Red Bull revenue.
- **Both weekends and January 2011 pricing raise the baseline**: weekends and post-January 2011 pricing shift daily Red Bull revenue upward, so evaluating promotions without those controls mixes traffic/price shifts with promo lift.
- **Multi-category and larger orders buy deals more often**: orders combining **Food + Drink + Hookah** or **Drink + Hookah**, larger multi-item baskets, and higher-value (`Q4`) orders show markedly higher deal attachment than single-item orders.

<a id="step-4-2"></a>
#### Step 4.2 - Data limitations
| Limitation | Why it matters | How we handle it |
|---|---|---|
| **Observational timing and promo format shift** | `RED BULL 2+1` / `3+2` ran on the old menu and shifted to `2`/`3`/`5 RED BULL` multi-packs in February–March 2011 alongside the January 2011 repricing | We include `post_jan1` and `day_orders` controls and inspect promo vs. non-promo cells directly |
| **Separate alcohol bills (`N` bills)** | Alcohol and food/hookah/Red Bull items in the same visit can appear on separate bills | We aggregate by `Bill Number` from `cafe_clean_lines.csv` and control for category flags |
| **No customer ID or cost data** | Repeat guests and unit margins are not observed in the till log | Results describe bill-level deal behavior and gross revenue (`Total`) only |